---
title: "Chapter 3: Functions"
---

[![View Source on GitHub](https://img.shields.io/badge/Source-GitHub-181717.svg?logo=github)](https://github.com/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/03-functions.ipynb)

The moment the same calculation appears in two places, copying it is the wrong answer: when the logic changes, you fix one copy and forget the other.

A function solves this. You write the logic once, give it a name, and call it from anywhere. Chapter 2 put your programs in motion with decisions and loops; this chapter adds naming so that motion can be reused. The energy theft detector you are building in Part 4 will need dozens of these named calculations: parse a meter reading, flag a threshold breach, compute a rolling mean. This chapter is where you learn to write them.

By the end of this chapter you will write functions that accept any number of arguments, use standard library modules, and return structured results. Chapter 4 (`04-classes.ipynb`) takes this further: it shows how to bundle functions and the data they operate on into a single named object, called a class.

::: {.callout-note collapse="true" icon=false}
## Learning objectives

By the end of Chapter 3 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Define functions with positional, default, and keyword-only parameters | Sec. 1 |
| 2 | Write Google-style docstrings | Sec. 1 |
| 3 | Use lambda expressions as sort keys and with `map()` / `filter()` | Sec. 2 |
| 4 | Accept variable arguments with `*args` and `**kwargs` | Sec. 3 |
| 5 | Use `math`, `json`, and `datetime` from the standard library | Sec. 4 |
:::

## 1. Functions

You need to compute a weighted grade. You could write the arithmetic inline every time you need it, but the day the weighting changes you have to find and update every copy.

A function wraps that arithmetic under a name. Write it once; call it anywhere.

![The anatomy of a Python function definition: keyword, name, typed parameters, defaults, and return type.](figs/function-anatomy.svg){fig-alt="Labeled diagram of a def statement showing the def keyword, function name, parameters with type hints, a default value, and the return type annotation."}

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: function anatomy</span><br><br>
<code>def name(param: type, param2: type = default) -&gt; return_type:</code><br><br>
<ul>
<li><code>def</code> declares the function</li>
<li><em>name</em> is what you call it by; use <code>snake_case</code></li>
<li>Parameters are inputs; a colon annotation declares the expected type</li>
<li><code>= default</code> makes a parameter optional; callers may omit it</li>
<li><code>-&gt; return_type</code> states what the function produces; omit for <code>None</code></li>
<li>The body is indented; <code>return</code> sends a value back to the caller</li>
</ul>
</div>

Start with the simplest possible function: one input, one output, nothing else:

In [ ]:
def greet(name: str) -> str:
    return f"Hello, {name}!"


print(greet("Alice"))
print(greet("Dr. Mwangi"))

Add a second parameter. A caller who provides both gets the explicit greeting; a caller who provides only the name gets a sensible default:

In [ ]:
def greet(name: str, greeting: str = "Hello") -> str:
    return f"{greeting}, {name}!"


print(greet("Alice"))
print(greet("Dr. Mwangi", greeting="Good morning"))

### Docstrings

A **docstring** is a string literal placed as the first statement of a function. It explains what the function does, what its inputs mean, and what it returns. IDEs, the built-in `help()`, and documentation generators all read it automatically.

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: Google-style docstrings</span><br><br>
The Google style is the most readable for short functions:<br><br>
<pre style='background:#F1F5F9;padding:8px;border-radius:4px;font-size:13px'>def fn(x: float, y: float) -&gt; float:
    """One-sentence summary.

    Args:
        x: What x means.
        y: What y means.

    Returns:
        What the function produces.
    """
    ...</pre>
</div>

Now build a real function using all four elements: multiple typed parameters, a default value, a docstring, and a return type.

This function calculates a weighted grade. The weights default to a 30/50/20 split but any caller can override them:

In [ ]:
def weighted_grade(
    midterm: float,
    final: float,
    project: float,
    weights: tuple[float, float, float] = (0.30, 0.50, 0.20),
) -> float:
    """Return the weighted average of three assessment scores.

    Args:
        midterm: Midterm exam score (0-100).
        final:   Final exam score (0-100).
        project: Project score (0-100).
        weights: Three weights summing to 1.0.

    Returns:
        Weighted average rounded to two decimal places, or 0.0 for invalid weights.
    """
    if abs(sum(weights) - 1.0) > 0.001:
        return 0.0
    return round(midterm * weights[0] + final * weights[1] + project * weights[2], 2)

Call with default weights, then override them. Keyword arguments make each call self-documenting:

In [ ]:
grade_default = weighted_grade(midterm=82.0, final=91.0, project=88.0)
grade_custom = weighted_grade(
    midterm=82.0,
    final=91.0,
    project=88.0,
    weights=(0.25, 0.50, 0.25),
)

print(f"Default weights : {grade_default}")
print(f"Custom weights  : {grade_custom}")

### Multiple return values

A function can return more than one value by packing them into a tuple. The caller unpacks them in one line:

In [ ]:
def score_summary(scores: list[float]) -> tuple[float, float, float, float]:
    """Return (mean, min, max, std) for a list of scores.

    Args:
        scores: List of numeric scores.

    Returns:
        Tuple of (mean, minimum, maximum, standard deviation).
        Returns four zeros for an empty list.
    """
    if not scores:
        return (0.0, 0.0, 0.0, 0.0)
    n = len(scores)
    mean = sum(scores) / n
    lo = min(scores)
    hi = max(scores)
    std = (sum((x - mean) ** 2 for x in scores) / n) ** 0.5
    return (mean, lo, hi, std)

Unpack the four return values into named variables in one assignment:

In [ ]:
exam_scores: list[float] = [78.0, 85.5, 92.0, 88.5, 95.0, 67.0, 81.0]
mean, lo, hi, std = score_summary(exam_scores)

print(f"Mean  : {mean:.1f}")
print(f"Range : {lo:.1f} to {hi:.1f}")
print(f"Std   : {std:.2f}")

### Normalisation

Z-score normalisation rescales each score by how many standard deviations it sits from the mean. The result is 0.0 for an exactly average score, positive above the mean, negative below.

In [ ]:
def normalize(value: float, mean: float, std: float) -> float:
    """Return the z-score of value given distribution parameters.

    Args:
        value: The raw score to normalise.
        mean:  Distribution mean.
        std:   Distribution standard deviation.

    Returns:
        Z-score, or 0.0 if std is zero.
    """
    if std == 0.0:
        return 0.0
    return (value - mean) / std

Compute the distribution parameters inline, then apply normalisation to every score:

In [ ]:
exam_scores: list[float] = [72.0, 85.0, 91.0, 68.0, 88.0, 77.0, 94.0, 63.0]

n = len(exam_scores)
mu = sum(exam_scores) / n
sig = (sum((x - mu) ** 2 for x in exam_scores) / n) ** 0.5

normalised = [normalize(s, mu, sig) for s in exam_scores]

for raw, z in zip(exam_scores, normalised, strict=False):
    bar = "#" * max(0, round((z + 3) * 5))
    print(f"  {raw:5.1f}  z={z:+.2f}  {bar}")

<div class='ark-mistake'>
<span class='ark-mistake-title'><i class="bi bi-bug-fill"></i> Common Mistake: mutable default arguments</span><br><br>
Never use a list, dict, or any other mutable object as a default parameter value. Python creates the default object <em>once</em> when the function is defined, not each time it is called. Every call that uses the default shares the same object.<br><br>
<strong>Wrong:</strong> <code>def add_score(score, history=[]):</code><br>
<strong>Right:</strong> use <code>None</code> as the sentinel and create the list inside the function body when <code>history is None</code>.
</div>

This is the bug in action: two independent calls share the same default list:

In [ ]:
def add_score_bad(score: float, history: list[float] = []) -> list[float]:  # noqa: B006
    history.append(score)
    return history


# These look independent but share the same default list
first = add_score_bad(88.0)
second = add_score_bad(91.0)

print(f"first  : {first}")  # [88.0, 91.0], not [88.0]
print(f"second : {second}")  # [88.0, 91.0], not [91.0]

Fix it by using `None` as the sentinel and creating a fresh list inside the function:

In [ ]:
def add_score(score: float, history: list[float] | None = None) -> list[float]:
    if history is None:
        history = []
    history.append(score)
    return history


first = add_score(88.0)
second = add_score(91.0)

print(f"first  : {first}")  # [88.0]
print(f"second : {second}")  # [91.0]

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: grade_letter</span><br><br>
Complete <code>grade_letter</code>: it takes a numeric score and returns the letter grade.<br><br>
Letter grades: A = 90+, B = 80-89, C = 70-79, D = 60-69, F = below 60.
</div>

In [ ]:
def grade_letter(score: float) -> str:
    """Return the letter grade for a numeric score."""
    return ...  # TODO: implement


print(grade_letter(92.0))  # expected: A
print(grade_letter(75.0))  # expected: C

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: classify_cohort</span><br><br>
Complete <code>classify_cohort</code>: it takes a list of scores and returns a <code>Counter</code> mapping each letter grade to the number of students who earned it. Use <code>grade_letter</code> from the cell above.
</div>

In [ ]:
from collections import Counter


def classify_cohort(scores: list[float]) -> Counter[str]:
    """Return a Counter mapping letter grades to student counts."""
    return ...  # TODO: implement


cohort = [88.0, 73.5, 55.0, 91.0, 67.0, 82.0, 95.5, 78.0, 61.0, 84.0]
distribution = classify_cohort(cohort)
if distribution is not ...:
    for grade in "ABCDF":
        print(f"{grade}: {distribution.get(grade, 0)}")

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: accept_login</span><br><br>
Complete <code>accept_login</code>: it checks a username and password against a dict of approved users and returns <code>True</code> if both match, <code>False</code> otherwise.
</div>

In [ ]:
def accept_login(users: dict[str, str], username: str, password: str) -> bool:
    """Return True if username exists and password matches.

    Args:
        users:    Mapping of username to hashed password.
        username: The submitted username.
        password: The submitted password.

    Returns:
        True on success, False on any mismatch.
    """
    return ...  # TODO: implement


approved: dict[str, str] = {"alice": "s3cr3t", "bob": "pa$$w0rd"}

if accept_login(approved, "alice", "s3cr3t") is not ...:
    print(accept_login(approved, "alice", "s3cr3t"))  # True
    print(accept_login(approved, "alice", "wrong"))  # False
    print(accept_login(approved, "carol", "s3cr3t"))  # False

## 2. Lambda functions

A **lambda** is an anonymous function defined in a single expression. It takes inputs on the left of `:` and produces a result on the right.

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: when to use a lambda</span><br><br>
Use a lambda when you need a throwaway function in one place: a sort key, a <code>map()</code> transform, or a <code>filter()</code> predicate. If the logic is more than one expression, or if you need it in more than one place, write a named function instead.
</div>

In [ ]:
# Lambda syntax: lambda params: expression
square = lambda x: x**2  # noqa: E731
print(f"square(5) = {square(5)}")

A lambda can take multiple parameters and include logic. `clamp` restricts a value to a range using `max` and `min`:

In [ ]:
clamp = lambda x, lo, hi: max(lo, min(x, hi))  # noqa: E731
print(f"clamp(115, 0, 100) = {clamp(115, 0, 100)}")

The most common use for lambdas is as a **sort key**: a function that extracts the comparison value from each element:

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "gpa": 3.95, "major": "CS"},
    {"name": "Bob", "gpa": 2.80, "major": "Math"},
    {"name": "Carol", "gpa": 3.40, "major": "Physics"},
    {"name": "Dan", "gpa": 3.70, "major": "CS"},
]

# Sort by GPA descending
by_gpa = sorted(students, key=lambda s: s["gpa"], reverse=True)

for s in by_gpa:
    print(f"{s['name']:<8} GPA={s['gpa']}")

### map() and filter()

`map(func, iterable)` applies `func` to every element and returns a lazy iterator. Wrap with `list()` to materialise the result:

In [ ]:
raw_scores: list[str] = ["78.5", "85.0", "92.3", "61.0", "88.7"]

# Convert strings to floats
scores: list[float] = list(map(float, raw_scores))
print(f"Parsed: {scores}")

`filter(func, iterable)` keeps only elements where `func` returns `True`. Equivalent to a comprehension with an `if` clause, but more expressive with a named predicate:

In [ ]:
# Keep only passing scores (>= 70)
passing: list[float] = list(filter(lambda s: s >= 70, scores))
print(f"Passing: {passing}")

# Same result as a comprehension
passing_comp = [s for s in scores if s >= 70]
print(f"Match:   {passing == passing_comp}")

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: top students by GPA</span><br><br>
Using <code>filter</code> and <code>sorted</code>, write a one-liner that returns the names of students with GPA >= 3.5, sorted by GPA descending.
</div>

In [ ]:
students: list[dict[str, object]] = [
    {"name": "Alice", "gpa": 3.95, "major": "CS"},
    {"name": "Bob", "gpa": 2.80, "major": "Math"},
    {"name": "Carol", "gpa": 3.40, "major": "Physics"},
    {"name": "Dan", "gpa": 3.70, "major": "CS"},
]

# TODO: one expression using filter + sorted
top: list[str] = ...

if top is not ...:
    print(top)  # ['Alice', 'Dan']

## 3. *args and **kwargs

A fixed signature only works when you know the number of arguments in advance. Sometimes the caller decides how many score lists to aggregate, or which attributes to attach to a student profile.

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: *args and **kwargs</span><br><br>
<code>*args</code> collects any number of <strong>positional</strong> arguments into a tuple.<br>
<code>**kwargs</code> collects any number of <strong>keyword</strong> arguments into a dict.<br><br>
Neither name is magic: <code>*scores</code> or <code>**attrs</code> work equally well. The <code>*</code> and <code>**</code> are what matter.
</div>

`*args` lets callers pass one list, three lists, or unpack a list of lists without changing the function signature:

In [ ]:
def aggregate_scores(*score_lists: list[float]) -> float:
    """Return the mean of all scores across any number of lists.

    Args:
        *score_lists: One or more lists of numeric scores.

    Returns:
        Overall mean, or 0.0 if no scores were provided.
    """
    all_scores = [s for lst in score_lists for s in lst]
    if not all_scores:
        return 0.0
    return round(sum(all_scores) / len(all_scores), 2)

Call with one list, multiple lists, or unpack with `*`. The function handles all three cases the same way:

In [ ]:
midterms: list[float] = [78.0, 85.0, 91.0]
finals: list[float] = [82.0, 88.0, 94.0]
projects: list[float] = [90.0, 87.0, 92.0]

print(aggregate_scores(midterms))
print(aggregate_scores(midterms, finals))
print(aggregate_scores(midterms, finals, projects))

`**kwargs` collects any number of keyword arguments into a dict. It is useful when you want to build a record from whatever attributes the caller provides:

In [ ]:
def build_student_profile(student_id: str, **attributes: object) -> dict[str, object]:
    """Build a student profile dict from keyword attributes.

    Args:
        student_id:   Unique student identifier.
        **attributes: Any additional fields to attach.

    Returns:
        Dict with student_id plus all provided attributes.
    """
    return {"student_id": student_id, **attributes}

Pass any combination of keyword arguments. `**` also unpacks a dict at the call site:

In [ ]:
p1 = build_student_profile("S1042", name="Alice", gpa=3.95, major="CS")
p2 = build_student_profile("S1043", name="Bob", gpa=2.80)

extra = {"program": "exchange", "cohort": 2024}
p3 = build_student_profile("S1044", name="Carol", **extra)

for p in (p1, p2, p3):
    print(p)

You can combine all four argument forms in one signature: fixed positional, `*args`, keyword-only (must be named at the call site), and `**kwargs`:

In [ ]:
def log_cohort_stats(
    semester: str,
    *courses: str,
    verbose: bool = False,
    **metrics: float,
) -> dict[str, object]:
    """Log and return stats for a semester cohort.

    Args:
        semester: Semester label, e.g. "2024-S1".
        *courses: Course codes included in this run.
        verbose:  If True, print each metric.
        **metrics: Named statistics to record.

    Returns:
        Dict with semester, courses, and metrics.
    """
    record: dict[str, object] = {
        "semester": semester,
        "courses": list(courses),
        "metrics": metrics,
    }
    if verbose:
        for key, val in metrics.items():
            print(f"  {key}: {val:.3f}")
    return record

`verbose` is keyword-only: it cannot be passed positionally because `*courses` comes before it in the signature:

In [ ]:
result = log_cohort_stats(
    "2024-S1",
    "CS101",
    "MATH201",
    verbose=True,
    pass_rate=0.821,
    mean_gpa=3.12,
    median_score=74.5,
)
print(result)

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: add a timestamp</span><br><br>
Extend <code>log_cohort_stats</code> to also include a <code>timestamp</code> key in the returned dict. Use <code>datetime.now(tz=UTC)</code> from the <code>datetime</code> module and format it with <code>.isoformat()</code>.
</div>

In [ ]:
from datetime import UTC, datetime


def log_cohort_stats_v2(
    semester: str,
    *courses: str,
    verbose: bool = False,
    **metrics: float,
) -> dict[str, object]:
    """Extended version: adds a UTC timestamp to the returned record."""
    return ...  # TODO: reuse log_cohort_stats logic and add timestamp


result = log_cohort_stats_v2("2024-S1", "CS101", pass_rate=0.821)
if result is not ...:
    print(result.get("timestamp", "missing"))

## 4. Modules and the standard library

A **module** is a Python file. When you write `import math`, Python loads `math.py` from the standard library and makes its contents available. No installation needed: these modules ship with every Python installation.

<div class='ark-concept'>
<span class='ark-concept-title'><i class="bi bi-key-fill"></i> Key Concept: importing from a module</span><br><br>
<ul>
<li><code>import math</code> gives you <code>math.sqrt()</code>, <code>math.pi</code>, etc.</li>
<li><code>from math import sqrt, pi</code> imports names directly; use this when you need just one or two items.</li>
<li>Choose whichever form makes the call site most readable.</li>
</ul>
</div>

The `math` module provides precise arithmetic for single numeric values:

In [ ]:
import math

print(f"pi           = {math.pi:.6f}")
print(f"e            = {math.e:.6f}")
print(f"sqrt(2)      = {math.sqrt(2):.6f}")
print(f"log10(1000)  = {math.log10(1000)}")
print(f"ceil(4.2)    = {math.ceil(4.2)}")
print(f"floor(4.8)   = {math.floor(4.8)}")
print(f"isnan(0.0)   = {math.isnan(0.0)}")
print(f"isinf(1e400) = {math.isinf(1e400)}")

`json` converts Python dicts, lists, and primitives to JSON strings and back. It is the standard format for saving run configs and logging result metadata:

In [ ]:
import json

run_result: dict[str, object] = {
    "run_id": "2024-S1-001",
    "course": "CS101",
    "pass_rate": 0.821,
    "mean_score": 74.5,
    "flags": ["regrade", "audit"],
}

serialised = json.dumps(run_result, indent=2)
print(serialised)

`json.loads` reverses the operation: it parses a JSON string back into a Python dict:

In [ ]:
# JSON string back to Python dict
loaded: dict[str, object] = json.loads(serialised)
print(f"pass_rate : {loaded['pass_rate']}")
print(f"flags     : {loaded['flags']}")

`datetime` represents a point in time. Always attach `timezone.utc` to avoid ambiguous naive datetime objects that can silently shift across time zones:

In [ ]:
now = datetime.now(tz=UTC)
print(f"Timestamp  : {now.isoformat()}")
print(f"Date part  : {now.strftime('%Y-%m-%d')}")
print(f"Year       : {now.year}")
print(f"Unix epoch : {now.timestamp():.0f}")

## Capstone exercises

These exercises use everything from sections 1-4. Each is self-contained.

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: student record processor</span><br><br>
Write a function <code>process_records(rows)</code> that takes a list of raw row dicts (each has <code>name</code>, <code>scores</code> as a list of floats, and <code>major</code>) and returns a list of summary dicts with keys: <code>name</code>, <code>major</code>, <code>mean</code>, <code>grade</code>.<br><br>
Use <code>score_summary</code> from Section 1 to get the mean, and <code>grade_letter</code> from Activity 1 to get the grade.
</div>

In [ ]:
def process_records(
    rows: list[dict[str, object]],
) -> list[dict[str, object]]:
    """Summarise a list of raw student rows.

    Args:
        rows: Each dict has "name", "scores", and "major".

    Returns:
        List of summary dicts with name, major, mean, grade.
    """
    return ...  # TODO: implement


students: list[dict[str, object]] = [
    {"name": "Alice", "scores": [88.0, 92.0, 85.0], "major": "CS"},
    {"name": "Bob", "scores": [62.0, 70.0, 58.0], "major": "Math"},
    {"name": "Carol", "scores": [91.0, 94.0, 89.0], "major": "Physics"},
    {"name": "Dan", "scores": [74.0, 68.0, 71.0], "major": "CS"},
]

result = process_records(students)
if result is not ...:
    for r in result:
        print(f"{r['name']:<8} {r['major']:<10} mean={r['mean']:.1f}  {r['grade']}")

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: moving window average</span><br><br>
Complete <code>moving_window_average</code>: it replaces each value with the mean of the values within <code>n_neighbors</code> positions on each side. Edge values use whatever neighbors are available.
</div>

In [ ]:
def moving_window_average(x: list[float], n_neighbors: int = 1) -> list[float]:
    """Replace each value with the mean of its n_neighbors on each side.

    Args:
        x:            Input list of numeric values.
        n_neighbors:  Window half-width.

    Returns:
        Smoothed list of the same length as x.
    """
    return ...  # TODO: implement


readings: list[float] = [36.5, 36.7, 36.8, 36.6, 36.9, 39.5, 36.7, 36.8]
smoothed = moving_window_average(readings, n_neighbors=2)
if smoothed is not ...:
    for raw, sm in zip(readings, smoothed, strict=False):
        print(f"{raw:.1f}  ->  {sm:.2f}")

## Further reading

| Resource | Why it matters |
|---|---|
| [PEP 484: Type Hints](https://peps.python.org/pep-0484/) | The original proposal; reading it explains why certain annotation rules exist |
| [PEP 257: Docstring Conventions](https://peps.python.org/pep-0257/) | The standard for what goes in a docstring |
| [Google Python Style Guide: Functions](https://google.github.io/styleguide/pyguide.html#383-functions-and-methods) | The docstring format used throughout this book |
| [Real Python: Lambda Functions](https://realpython.com/python-lambda/) | Covers every edge case, including when NOT to use lambdas |
| [Python `functools` docs](https://docs.python.org/3/library/functools.html) | `partial`, `lru_cache`, `reduce`: the tools that build on what this chapter covers |

## Summary

| Concept | Key rule |
|---|---|
| Functions | Annotate all params and return types; use Google-style docstrings |
| Defaults | Never use a mutable object as a default; use `None` and create inside |
| Multiple returns | Pack into a tuple; unpack with `a, b = fn()` |
| Lambda | One expression, one place; write a named function for anything more |
| `*args` | Collects positional arguments into a tuple |
| `**kwargs` | Collects keyword arguments into a dict |
| Standard library | `math` for precise arithmetic; `json` for serialisation; `datetime` for timestamps |

**Next:** [Chapter 4: Classes and patterns](04-classes.ipynb): how to bundle functions and the data they operate on into a single named object.